# 03 - Test & Demo

Bu notebook:
1) Preprocess edilmiş `.npz` dosyasından test setini yükler
2) Eğitilmiş modeli yükler
3) Test metriklerini hesaplar (Accuracy / Precision / Recall / F1)
4) İstersen tek bir görüntüyle demo prediction yapar

> Yanlış sonuçların en sık sebebi: preprocess'te kullanılan `class_names` ile testteki label mapping'in farklı olması. Bu notebook mapping'i direkt `.npz` içinden alır.

In [ ]:

import os
from pathlib import Path
import numpy as np

PREPROCESSED_DIR = Path("./preprocessed")
MODEL_DIR = Path("./models")

SEED = 42
np.random.seed(SEED)

print("PREPROCESSED_DIR:", PREPROCESSED_DIR.resolve())
print("MODEL_DIR:", MODEL_DIR.resolve())


## 1) Dosya seçimi

In [ ]:

DATASET_FILE = PREPROCESSED_DIR / "plant_disease_detection_preprocessed.npz"  # change if needed
OUT_PREFIX = DATASET_FILE.stem.replace("_preprocessed","")

data = np.load(DATASET_FILE, allow_pickle=True)
X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"].astype(np.int64)
class_names = list(data["class_names"])
num_classes = len(class_names)

print("Loaded:", DATASET_FILE)
print("test:", X_test.shape, "classes:", num_classes)


## 2) Metrikler + Confusion Matrix

In [ ]:

def accuracy(y_true, y_pred):
    return float((y_true == y_pred).mean())

def macro_precision_recall_f1(y_true, y_pred, num_classes):
    eps = 1e-12
    precisions, recalls, f1s = [], [], []
    for c in range(num_classes):
        tp = int(((y_true == c) & (y_pred == c)).sum())
        fp = int(((y_true != c) & (y_pred == c)).sum())
        fn = int(((y_true == c) & (y_pred != c)).sum())
        p = tp / (tp + fp + eps)
        r = tp / (tp + fn + eps)
        f1 = 2 * p * r / (p + r + eps)
        precisions.append(p); recalls.append(r); f1s.append(f1)
    return float(np.mean(precisions)), float(np.mean(recalls)), float(np.mean(f1s))

def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm


## 3) Model yükleme ve test

`MODEL_TYPE` seç:
- `logreg_numpy`
- `logreg_torch`
- `random_forest` (eğer eğitimde kaydettiysen)


In [ ]:

MODEL_TYPE = "logreg_numpy"  # "logreg_numpy" | "logreg_torch" | "random_forest"

if MODEL_TYPE == "logreg_numpy":
    model_path = MODEL_DIR / f"{OUT_PREFIX}_logreg_numpy.npz"
    m = np.load(model_path, allow_pickle=True)
    W = m["W"].astype(np.float32)
    b = m["b"].astype(np.float32)

    logits = X_test @ W + b
    y_pred = logits.argmax(axis=1)

elif MODEL_TYPE == "logreg_torch":
    import torch
    model_path = MODEL_DIR / f"{OUT_PREFIX}_logreg_torch.pt"
    payload = torch.load(model_path, map_location="cpu")
    state = payload["state_dict"]

    # simple forward
    W = state["weight"].numpy().T  # torch Linear: (C, D) -> transpose
    b = state["bias"].numpy().reshape(1, -1)

    logits = X_test @ W + b
    y_pred = logits.argmax(axis=1)

elif MODEL_TYPE == "random_forest":
    import joblib
    model_path = MODEL_DIR / f"{OUT_PREFIX}_random_forest.joblib"
    payload = joblib.load(model_path)
    rf = payload["model"]
    y_pred = rf.predict(X_test)

else:
    raise ValueError("Unknown MODEL_TYPE")

acc = accuracy(y_test, y_pred)
p, r, f1 = macro_precision_recall_f1(y_test, y_pred, num_classes)

print("Test Accuracy :", acc)
print("Test Precision:", p)
print("Test Recall   :", r)
print("Test F1-score :", f1)

cm = confusion_matrix_np(y_test, y_pred, num_classes)
print("Confusion Matrix shape:", cm.shape)


## 4) (Opsiyonel) Confusion Matrix'i görselleştir

In [ ]:

SHOW_CM = False

if SHOW_CM:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


## 5) (Opsiyonel) Tek görüntüyle demo

Bu kısım, preprocess ile **aynı** dönüşümü uygular (resize + grayscale + normalize + standardize) ve sınıf tahmini verir.

In [ ]:

from PIL import Image

DEMO_IMAGE_PATH = None  # e.g. "../Dataset/plant_disease_detection/Test/SomeClass/img.jpg"

def preprocess_single_image(img_path, data_npz):
    img_size = int(data_npz["img_size"])
    grayscale = bool(int(data_npz["grayscale"]))
    mean = data_npz["mean"].astype(np.float32)
    std  = data_npz["std"].astype(np.float32)

    img = Image.open(img_path)
    img = img.convert("L" if grayscale else "RGB")
    img = img.resize((img_size, img_size))
    x = np.asarray(img, dtype=np.float32) / 255.0
    x = x.reshape(1, -1).astype(np.float32)
    x = (x - mean) / std
    return x

if DEMO_IMAGE_PATH is not None:
    x = preprocess_single_image(DEMO_IMAGE_PATH, data)
    if MODEL_TYPE in ["logreg_numpy", "logreg_torch"]:
        logits = x @ W + b
        pred = int(logits.argmax(axis=1)[0])
    elif MODEL_TYPE == "random_forest":
        pred = int(rf.predict(x)[0])

    print("Predicted class:", class_names[pred])
